# Preprocess `00_Datasets.txt` for **PU learning**

This replaces Karin's `00_new_preprocessing_karin.ipynb`.

**What's different**: Karin filters out users with `IP_check=False` and ends up with a binary classification problem. We *keep* them as the unlabeled set `U`, producing a proper PU setup.

**Three-way label `label_3way`**:
- `P` — IP-checked AND user_sameIP is non-empty → confirmed sockpuppet
- `N` — IP-checked AND user_sameIP is empty → high-confidence not-sockpuppet (validation holdout)
- `U` — IP-unverifiable → genuinely unlabeled (where hidden sockpuppets may live)

For the PU classifier we use **P vs U** as the training signal. **N** is held out as a clean validation set: if our PU model classifies these N users as positive, that's a false-positive rate we can actually measure.

**Output**: `user_features_pu.csv` — one row per user, ready for the PU pipeline.


In [1]:
import json, re, ast, math
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict

# --- CONFIG ---
INPUT_TXT  = "../00_Datasets.txt"   # path to the raw file
OUTPUT_CSV = "./user_features_pu.csv"
GROUPS_SIDECAR = "./user_groups_sidecar_pu.csv"
TAG_FILTER = "การเมือง"                         # keep posts tagged politics
# --------------

## Step 1: Parse the raw JSON

Handle BOM, control characters, and any truncation — same robust loading as Karin's notebook.


In [2]:
with open(INPUT_TXT, 'r', encoding='utf-8-sig') as f:
    raw = f.read().strip()

raw = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', raw)
if raw.endswith(','):
    raw = raw[:-1] + '\n]'
elif not raw.endswith(']'):
    raw = raw + '\n]'

records = json.loads(raw, strict=False)
print(f'Loaded {len(records):,} raw post records')

Loaded 95,904 raw post records


## Step 2: Apply tag filter (same as Karin)

Keep only politics posts.


In [3]:
posts = [
    r for r in records
    if isinstance(r.get('tag'), list) and TAG_FILTER in r['tag']
]
print(f'After tag filter ({TAG_FILTER!r}): {len(posts):,} posts')

# Free the big raw list
del records

After tag filter ('การเมือง'): 13,482 posts


## Step 3: Three-way user labeling

For each user, look at *all* their posts to decide their label.

- If **any** post has `IP_check=True` and **any** has non-empty `user_sameIP` → **P** (confirmed sockpuppet)
- Else if **any** post has `IP_check=True` (and none flagged sharing) → **N** (IP-checked clean)
- Else (no post was IP-checked) → **U** (unverifiable, unlabeled)


In [4]:
user_ipcheck  = defaultdict(list)
user_sameip   = defaultdict(list)

for r in posts:
    uid = r['user_id']
    user_ipcheck[uid].append(bool(r.get('IP_check', False)))
    sip = r.get('user_sameIP', [])
    if isinstance(sip, list):
        user_sameip[uid].append(len(sip) > 0)
    else:
        user_sameip[uid].append(False)

def classify(uid):
    any_checked = any(user_ipcheck[uid])
    any_shared  = any(user_sameip[uid])
    if not any_checked:
        return 'U'
    return 'P' if any_shared else 'N'

user_label = {uid: classify(uid) for uid in user_ipcheck}

label_counts = Counter(user_label.values())
print('Three-way label distribution:')
for lbl in ('P', 'N', 'U'):
    n = label_counts.get(lbl, 0)
    print(f'  {lbl}: {n:5d}  ({n/sum(label_counts.values())*100:.1f}%)')
print(f'  TOTAL: {sum(label_counts.values()):5d}')

Three-way label distribution:
  P:   142  (13.2%)
  N:   252  (23.5%)
  U:   678  (63.2%)
  TOTAL:  1072


## Step 4: Sockpuppet group IDs (for P users only)

Connected components over the IP-sharing graph, same as Karin's notebook.


In [5]:
import networkx as nx

G = nx.Graph()
for r in posts:
    if r.get('IP_check') is not True:
        continue
    uid = r['user_id']
    sip = r.get('user_sameIP', [])
    if not (isinstance(sip, list) and sip):
        continue
    G.add_node(uid)
    for other in sip:
        G.add_edge(uid, other)

user_to_group = {}
for gid, comp in enumerate(nx.connected_components(G)):
    for u in comp:
        user_to_group[u] = gid

print(f'Sockpuppet groups: {len(set(user_to_group.values()))}')
print(f'Users in groups:   {len(user_to_group)}')

Sockpuppet groups: 74
Users in groups:   283


## Step 5: Build a per-post DataFrame

We need this to compute behavioral features per user.


In [6]:
df = pd.DataFrame(posts)
print(f'DataFrame: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

# Datetime
df['dt'] = pd.to_datetime(df['datetime'], format='%m/%d/%Y %H:%M:%S', errors='coerce')
print(f'Bad datetimes: {df["dt"].isna().sum()}')
df = df.sort_values(['user_id', 'dt']).reset_index(drop=True)

# Parse profile_like / emo_like list strings if they're strings (sometimes raw JSON keeps them as lists)
def to_list(v):
    if isinstance(v, list): return v
    if pd.isna(v): return []
    try:
        x = ast.literal_eval(v)
        return x if isinstance(x, list) else [x]
    except Exception:
        return []

df['profile_like']  = df['profile_like'].apply(to_list)
df['n_reacts']      = df['profile_like'].apply(len)

DataFrame: (13482, 21)
Columns: ['num_0', 'num_1', 'num_r', 'thread_id', 'thread_name', 'user_name', 'user_id', 'messages', 'length', 'helpful', 'forum', 'tag', 'datetime', 'profile_like', 'emo_like', 'user_type', 'user_sameIP', 'IP', 'IP_check', 'link', 'messages_ref']
Bad datetimes: 0


## Step 6: Compute behavioral features

Same families as before: activity, temporal, content, engagement, topic, cross-thread coordination.


In [7]:
def entropy(counts):
    total = sum(counts)
    if total == 0: return 0.0
    probs = [c/total for c in counts if c > 0]
    return -sum(p * math.log2(p) for p in probs)

def burstiness(intervals_sec):
    if len(intervals_sec) < 2: return 0.0
    mu = float(np.mean(intervals_sec))
    sigma = float(np.std(intervals_sec))
    if mu + sigma == 0: return 0.0
    return (sigma - mu) / (sigma + mu)

def char_entropy(text):
    if not isinstance(text, str) or len(text) == 0: return 0.0
    return entropy(list(Counter(text).values()))


# --- Activity ---
activity = df.groupby('user_id').agg(
    n_posts=('thread_id', 'size'),
    n_threads=('thread_id', 'nunique'),
).reset_index()
span = df.groupby('user_id')['dt'].agg(['min','max']).reset_index()
span['days_active'] = (span['max'] - span['min']).dt.total_seconds() / 86400
activity = activity.merge(span[['user_id','days_active']], on='user_id')
activity['posts_per_thread'] = activity['n_posts'] / activity['n_threads']
activity['posts_per_day']    = activity['n_posts'] / activity['days_active'].clip(lower=1/24)

# --- Temporal ---
temporal_records = []
for uid, g in df.groupby('user_id'):
    times = g['dt'].sort_values().tolist()
    hours = [t.hour for t in times]
    dows  = [t.dayofweek for t in times]
    if len(times) >= 2:
        intervals = [(times[i+1] - times[i]).total_seconds() for i in range(len(times)-1)]
        mean_int  = float(np.mean(intervals))
        median_int = float(np.median(intervals))
        min_int   = float(np.min(intervals))
        std_int   = float(np.std(intervals))
        B         = burstiness(intervals)
    else:
        mean_int = median_int = min_int = std_int = B = 0.0
    temporal_records.append({
        'user_id': uid,
        'hour_entropy_bits':      entropy(list(Counter(hours).values())),
        'dayofweek_entropy_bits': entropy(list(Counter(dows).values())),
        'interpost_mean_sec':     mean_int,
        'interpost_median_sec':   median_int,
        'interpost_min_sec':      min_int,
        'interpost_std_sec':      std_int,
        'burstiness':             B,
        'frac_posts_night':       sum(1 for h in hours if h < 6 or h >= 22) / max(len(hours), 1),
    })
temporal = pd.DataFrame(temporal_records)

# --- Content ---
df['msg_str']      = df['messages'].fillna('').astype(str)
df['msg_char_ent'] = df['msg_str'].apply(char_entropy)
content = df.groupby('user_id').agg(
    msg_len_mean=('length','mean'),
    msg_len_std=('length','std'),
    msg_len_min=('length','min'),
    msg_len_max=('length','max'),
    msg_len_median=('length','median'),
    msg_char_ent_mean=('msg_char_ent','mean'),
    msg_char_ent_std=('msg_char_ent','std'),
).reset_index().fillna(0)

# --- Engagement ---
engagement = df.groupby('user_id').agg(
    num0_mean=('num_0','mean'),
    num0_sum=('num_0','sum'),
    num1_mean=('num_1','mean'),
    num1_sum=('num_1','sum'),
    reacts_mean=('n_reacts','mean'),
    reacts_sum=('n_reacts','sum'),
).reset_index()
engagement['reacts_per_post'] = engagement['reacts_sum'] / df.groupby('user_id').size().values

# --- Topic ---
topic_records = []
for uid, g in df.groupby('user_id'):
    all_tags = [t for tags in g['tag'] for t in (tags if isinstance(tags, list) else [])]
    tc = Counter(all_tags)
    topic_records.append({
        'user_id': uid,
        'n_unique_tags':    len(tc),
        'tag_entropy_bits': entropy(list(tc.values())),
        'total_tag_uses':   sum(tc.values()),
    })
topics = pd.DataFrame(topic_records)

# --- Cross-thread coordination ---
coord_records = []
for uid, g in df.groupby('user_id'):
    tc = g['thread_id'].value_counts()  # Series
    coord_records.append({
        'user_id': uid,
        'thread_entropy_bits':       entropy(list(tc)),
        'max_posts_in_thread':       int(tc.max()),
        'mean_posts_per_thread':     float(tc.mean()),
        'frac_threads_with_one_post': float((tc == 1).mean()),
    })
coord = pd.DataFrame(coord_records)

print('Feature blocks built.')

Feature blocks built.


## Step 7: Assemble final user-level table


In [8]:
# Per-user labels
user_ids = sorted(user_label.keys())
labels_df = pd.DataFrame({
    'user_id': user_ids,
    'label_3way': [user_label[u] for u in user_ids],
})
labels_df['is_sockpuppet'] = (labels_df['label_3way'] == 'P').astype(int)
labels_df['ip_checked']    = labels_df['label_3way'].isin(['P', 'N']).astype(int)
labels_df['sockpuppet_group_id'] = labels_df['user_id'].map(user_to_group).fillna(-1).astype(int)

user_features = (
    labels_df
    .merge(activity,   on='user_id')
    .merge(temporal,   on='user_id')
    .merge(content,    on='user_id')
    .merge(engagement, on='user_id')
    .merge(topics,     on='user_id')
    .merge(coord,      on='user_id')
)

print(f'Final shape: {user_features.shape}')
print(f'Label breakdown:')
print(user_features['label_3way'].value_counts())

Final shape: (1072, 39)
Label breakdown:
label_3way
U    678
N    252
P    142
Name: count, dtype: int64


## Step 8: Quick separability check (P vs N vs U)

Three-way mean comparison. If features behave sensibly, we expect:
- **P** = sockpuppet-like (high burstiness, varied hours, many threads)
- **N** = boring-user-like (low burstiness, narrow hours)
- **U** = somewhere in between (because U mixes hidden positives with true negatives)


In [9]:
feature_cols = [c for c in user_features.columns
                if c not in ('user_id','label_3way','is_sockpuppet',
                             'ip_checked','sockpuppet_group_id')]

means_3way = user_features.groupby('label_3way')[feature_cols].mean().T
means_3way = means_3way[['P', 'N', 'U']] if all(c in means_3way.columns for c in ['P','N','U']) else means_3way

# Where does U sit between P and N? Closer to P suggests hidden sockpuppets.
def position(row):
    P, N, U = row.get('P'), row.get('N'), row.get('U')
    if P is None or N is None or U is None: return np.nan
    if P == N: return np.nan
    return (U - N) / (P - N)  # 0 = behaves like N, 1 = behaves like P

means_3way['U_position_between_N_and_P'] = means_3way.apply(position, axis=1)

print('Mean feature values by 3-way label, and where U sits between N and P:')
print(means_3way.round(3).to_string())
print()
print('Features where U behaves more like P than like N (U_position > 0.5):')
sus = means_3way[means_3way['U_position_between_N_and_P'] > 0.5]
print(sus[['P','N','U','U_position_between_N_and_P']].round(3).to_string())

Mean feature values by 3-way label, and where U sits between N and P:
label_3way                          P           N           U  U_position_between_N_and_P
n_posts                        16.831      12.171      11.836                      -0.072
n_threads                      11.979       9.365       8.602                      -0.292
days_active                     8.790       6.831       7.065                       0.119
posts_per_thread                1.344       1.271       1.259                      -0.170
posts_per_day                  13.027      16.531      15.363                       0.333
hour_entropy_bits               1.563       0.988       1.019                       0.053
dayofweek_entropy_bits          1.092       0.751       0.769                       0.053
interpost_mean_sec          87543.278  138223.319  124040.870                       0.280
interpost_median_sec        58144.634  117400.240   96741.525                       0.349
interpost_min_sec           26

## Step 9: Save

Two files:
- `user_features_pu.csv` — for the PU pipeline (has `label_3way`, `is_sockpuppet`)
- `user_groups_sidecar_pu.csv` — group IDs, kept separate to avoid label leakage


In [10]:
groups_sidecar = user_features[['user_id','sockpuppet_group_id']].copy()
groups_sidecar.to_csv(GROUPS_SIDECAR, index=False)

user_features_clean = user_features.drop(columns=['sockpuppet_group_id'])
user_features_clean.to_csv(OUTPUT_CSV, index=False)

print(f'Saved {len(user_features_clean)} users to {OUTPUT_CSV}')
print(f'  - P (confirmed sockpuppet): {(user_features_clean.label_3way == "P").sum()}')
print(f'  - N (confirmed clean):       {(user_features_clean.label_3way == "N").sum()}')
print(f'  - U (unlabeled):             {(user_features_clean.label_3way == "U").sum()}')
print(f'Saved group sidecar to {GROUPS_SIDECAR}')
print()
print('Next: in pu_pipeline_v2.ipynb, the script will use P vs U for training and')
print('hold N out as a clean validation set for measuring false positives.')

Saved 1072 users to ./user_features_pu.csv
  - P (confirmed sockpuppet): 142
  - N (confirmed clean):       252
  - U (unlabeled):             678
Saved group sidecar to ./user_groups_sidecar_pu.csv

Next: in pu_pipeline_v2.ipynb, the script will use P vs U for training and
hold N out as a clean validation set for measuring false positives.
